# Qwen2.5-7B-Instruct Naked Canonical Examples

Minimal notebook for direct generation on the same 9 canonical prompts, without ComplianceGPT retrieval or contract logic.

Outputs are written to:

`/content/drive/MyDrive/ComplianceGPT/src/3_canonical_examples/results`


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import pandas as pd
import torch
from datetime import datetime
from IPython.display import display
from transformers import AutoTokenizer, AutoModelForCausalLM

RESULTS_DIR = "/content/drive/MyDrive/ComplianceGPT/src/3_canonical_examples/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True,
)

print("[OK] Model loaded:", MODEL_ID)
print("RESULTS_DIR:", RESULTS_DIR)


Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

[OK] Model loaded: Qwen/Qwen2.5-7B-Instruct
RESULTS_DIR: /content/drive/MyDrive/ComplianceGPT/src/3_canonical_examples/results


In [2]:
CANONICAL_EXAMPLES = {
    "odp_password_length": [
        "What is the required password length in NIST SP 800-53?",
        "Under NIST SP 800-53, how long do passwords need to be?",
        "Based on NIST SP 800-53, what password length is required?",
    ],
    "language_mismatch_mfa_admins": [
        "Do we need MFA for admins under NIST SP 800-53?",
        "Under NIST SP 800-53, do administrators need multi-factor authentication?",
        "Do privileged users have to use MFA in NIST SP 800-53?",
    ],
    "posthoc_citation_privileged_mfa": [
        "Do privileged users need multi-factor authentication? Please include the control citation.",
        "Which NIST SP 800-53 control says privileged users need MFA? Please answer with the citation.",
        "Please answer using NIST SP 800-53 and include the control citation: Do admins need MFA?",
    ],
}

print("[OK] Canonical examples loaded.")
for k, v in CANONICAL_EXAMPLES.items():
    print(f"- {k}: {len(v)} prompts")


[OK] Canonical examples loaded.
- odp_password_length: 3 prompts
- language_mismatch_mfa_admins: 3 prompts
- posthoc_citation_privileged_mfa: 3 prompts


In [3]:
def generate_answer(question: str, max_new_tokens: int = 384) -> str:
    messages = [{"role": "user", "content": question}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    new_tokens = generated_ids[:, model_inputs.input_ids.shape[1]:]
    return tokenizer.batch_decode(new_tokens, skip_special_tokens=True)[0]


In [4]:
rows = []

for example_name, prompts in CANONICAL_EXAMPLES.items():
    print("\n" + "=" * 100)
    print(f"EXAMPLE: {example_name}")
    print("=" * 100)
    for variant_idx, prompt in enumerate(prompts, start=1):
        print(f"\n[{variant_idx}/{len(prompts)}] {prompt}")
        answer = generate_answer(prompt)
        print("answer:")
        print(answer)

        rows.append({
            "model_id": MODEL_ID,
            "example_name": example_name,
            "variant_idx": variant_idx,
            "prompt": prompt,
            "answer_text": answer,
        })

df_results = pd.DataFrame(rows)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path = os.path.join(RESULTS_DIR, f"qwen_naked_canonical_examples_{timestamp}.csv")
json_path = os.path.join(RESULTS_DIR, f"qwen_naked_canonical_examples_{timestamp}.json")
latest_csv_path = os.path.join(RESULTS_DIR, "qwen_naked_canonical_examples_latest.csv")
latest_json_path = os.path.join(RESULTS_DIR, "qwen_naked_canonical_examples_latest.json")

df_results.to_csv(csv_path, index=False)
df_results.to_csv(latest_csv_path, index=False)

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)
with open(latest_json_path, "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)

print("\n[OK] Batch run complete. Rows:", len(df_results))
print("[OK] Saved CSV:", csv_path)
print("[OK] Saved JSON:", json_path)

display(df_results)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



EXAMPLE: odp_password_length

[1/3] What is the required password length in NIST SP 800-53?
answer:
NIST Special Publication (SP) 800-53 provides guidelines for security and privacy controls in information systems. Regarding password length, NIST SP 800-53 does not specify an exact minimum length but emphasizes the importance of complexity and length together.

In particular, NIST recommends that passwords should be long enough to resist brute-force attacks while also being manageable for users. The publication suggests that a minimum length of 8 characters is often insufficient, especially when considering the increasing power of modern computing. Instead, it recommends a minimum length of at least 12 characters, with a preference for longer passwords.

The key focus in NIST's guidance is on the combination of length and complexity rather than just one factor. This means that while a longer password is better, it should also include a mix of character types (uppercase letters, lowerc

,model_id,example_name,variant_idx,prompt,answer_text
0,Qwen/Qwen2.5-7B-Instruct,odp_password_length,1,What is the required password length in NIST S...,NIST Special Publication (SP) 800-53 provides ...
1,Qwen/Qwen2.5-7B-Instruct,odp_password_length,2,"Under NIST SP 800-53, how long do passwords ne...",According to NIST Special Publication (SP) 800...
2,Qwen/Qwen2.5-7B-Instruct,odp_password_length,3,"Based on NIST SP 800-53, what password length ...",According to NIST Special Publication (SP) 800...
3,Qwen/Qwen2.5-7B-Instruct,language_mismatch_mfa_admins,1,Do we need MFA for admins under NIST SP 800-53?,Under the NIST Special Publication (SP) 800-53...
4,Qwen/Qwen2.5-7B-Instruct,language_mismatch_mfa_admins,2,"Under NIST SP 800-53, do administrators need m...",According to the National Institute of Standar...
5,Qwen/Qwen2.5-7B-Instruct,language_mismatch_mfa_admins,3,Do privileged users have to use MFA in NIST SP...,According to the NIST Special Publication 800-...
6,Qwen/Qwen2.5-7B-Instruct,posthoc_citation_privileged_mfa,1,Do privileged users need multi-factor authenti...,"Yes, privileged users should be required to us..."
7,Qwen/Qwen2.5-7B-Instruct,posthoc_citation_privileged_mfa,2,Which NIST SP 800-53 control says privileged u...,The NIST Special Publication 800-53 revision 4...
8,Qwen/Qwen2.5-7B-Instruct,posthoc_citation_privileged_mfa,3,Please answer using NIST SP 800-53 and include...,"According to NIST Special Publication 800-53, ..."
